# DiffusionNet Starter — Smoke Test trên Kaggle

**Mục tiêu:** xác nhận môi trường + DiffusionNet chạy được end-to-end trên 1 mesh đơn giản.

Sau khi notebook này chạy xanh, bạn đã sẵn sàng nạp dataset thật (Breaking Bad) vào để train.

**Trước khi chạy — Settings (panel bên phải):**
- Accelerator: **GPU T4 x2** (hoặc P100)
- Internet: **ON**
- Persistence: **Files only**



## 1. Cài đặt thư viện

In [ ]:
# Cài đặt các thư viện cần cho mesh + DiffusionNet
!pip install -q trimesh potpourri3d robust_laplacian plyfile plotly

## 2. Clone DiffusionNet và add vào Python path

In [ ]:
import os, sys

# Clone repo nếu chưa có
if not os.path.exists('/kaggle/working/diffusion-net'):
    !git clone https://github.com/nmwsharp/diffusion-net.git /kaggle/working/diffusion-net

# Thêm src vào sys.path để có thể import
sys.path.append('/kaggle/working/diffusion-net/src')
print('DiffusionNet đã sẵn sàng để import.')

## 3. Kiểm tra môi trường (GPU, version)

In [ ]:
import torch
import numpy as np
import trimesh
import diffusion_net

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
print(f'NumPy version   : {np.__version__}')
print(f'Trimesh version : {trimesh.__version__}')

## 4. Tải / tạo 1 mesh mẫu

Mình dùng `icosphere` của trimesh để khỏi phụ thuộc file ngoài. Sau này bạn sẽ thay bằng mesh từ Breaking Bad Dataset (file `.obj` hoặc `.ply`).

In [ ]:
# Tạo 1 mesh hình cầu — đủ đơn giản để test, đủ phức tạp để DiffusionNet hoạt động
mesh = trimesh.creation.icosphere(subdivisions=4)

verts_np = np.asarray(mesh.vertices, dtype=np.float32)
faces_np = np.asarray(mesh.faces, dtype=np.int64)

print(f'Số vertices : {verts_np.shape[0]}')   # ~2562
print(f'Số faces    : {faces_np.shape[0]}')   # ~5120
print(f'Vertices shape: {verts_np.shape}')
print(f'Faces shape   : {faces_np.shape}')

# Khi load từ file thật:
# mesh = trimesh.load('/kaggle/input/your-dataset/some_mesh.obj')
# verts_np = np.asarray(mesh.vertices, dtype=np.float32)
# faces_np = np.asarray(mesh.faces, dtype=np.int64)

## 5. Visualize mesh (interactive 3D ngay trong notebook)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Mesh3d(
    x=verts_np[:, 0], y=verts_np[:, 1], z=verts_np[:, 2],
    i=faces_np[:, 0], j=faces_np[:, 1], k=faces_np[:, 2],
    color='lightblue', opacity=0.85, flatshading=True,
)])
fig.update_layout(
    title='Mesh mẫu — kéo chuột để xoay',
    scene=dict(aspectmode='data'),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

## 6. Tính các operator hình học DiffusionNet cần

Hàm `get_operators` sẽ tính: mass matrix, Laplacian, eigenvalues/vectors, spatial gradients. Lần đầu chậm vì phải tính, lần sau nhanh vì cache.

**Bạn không cần hiểu toán bên trong** — coi nó như preprocessing tự động.

In [ ]:
# Convert numpy -> torch
verts = torch.tensor(verts_np, dtype=torch.float32)
faces = torch.tensor(faces_np, dtype=torch.long)

# Tạo thư mục cache cho operators (tính 1 lần, dùng nhiều lần)
OP_CACHE = '/kaggle/working/op_cache'
os.makedirs(OP_CACHE, exist_ok=True)

# Tính operators
frames, mass, L, evals, evecs, gradX, gradY = diffusion_net.geometry.get_operators(
    verts, faces,
    k_eig=128,                  # số eigenvalues — 128 là default tốt
    op_cache_dir=OP_CACHE,
)

print(f'mass  shape: {mass.shape}')
print(f'L     shape: {L.shape}  (sparse Laplacian)')
print(f'evals shape: {evals.shape}')
print(f'evecs shape: {evecs.shape}')
print(f'gradX shape: {gradX.shape}')
print(f'gradY shape: {gradY.shape}')

## 7. Tạo mô hình DiffusionNet

Cấu hình cho bài toán của bạn:
- `C_in=3`: input là tọa độ xyz (sau này có thể đổi sang HKS)
- `C_out=2`: 2 lớp **fracture** vs **original**
- `C_width=64`: width ẩn — nhỏ để chạy nhanh khi smoke test, lên 128 khi train thật
- `N_block=4`: 4 diffusion blocks
- `outputs_at='vertices'`: prediction per-vertex (đúng kiểu segmentation)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = diffusion_net.layers.DiffusionNet(
    C_in=3,
    C_out=2,
    C_width=64,
    N_block=4,
    outputs_at='vertices',
    dropout=True,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Số tham số mô hình: {n_params:,}')
print(model)

## 8. Forward pass thử (kiểm tra mô hình chạy được)

In [ ]:
# Chuyển tất cả lên device
verts_d = verts.to(device)
faces_d = faces.to(device)
mass_d  = mass.to(device)
L_d     = L.to(device)
evals_d = evals.to(device)
evecs_d = evecs.to(device)
gradX_d = gradX.to(device)
gradY_d = gradY.to(device)

# Input feature = xyz của vertices
feats = verts_d

model.eval()
with torch.no_grad():
    logits = model(
        x_in=feats,
        mass=mass_d, L=L_d,
        evals=evals_d, evecs=evecs_d,
        gradX=gradX_d, gradY=gradY_d,
        faces=faces_d,
    )

print(f'Output shape: {logits.shape}  (kỳ vọng: [N_vertices, 2])')
print(f'Sample logits 5 vertex đầu:')
print(logits[:5].cpu().numpy())

## 9. Mini training step — kiểm tra backward + loss

Dùng nhãn random để xem loss có giảm khi train không (chỉ để kiểm tra cơ chế hoạt động, không có ý nghĩa thực tế).

In [ ]:
import torch.nn.functional as F

# Nhãn giả: gán random 0/1 cho mỗi vertex
labels = torch.randint(0, 2, (verts.shape[0],), device=device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model.train()

for step in range(20):
    optimizer.zero_grad()
    logits = model(
        x_in=feats,
        mass=mass_d, L=L_d,
        evals=evals_d, evecs=evecs_d,
        gradX=gradX_d, gradY=gradY_d,
        faces=faces_d,
    )
    loss = F.cross_entropy(logits, labels)
    loss.backward()
    optimizer.step()
    if step % 5 == 0:
        print(f'Step {step:2d} | loss = {loss.item():.4f}')

print('\nMô hình train được. Mọi thứ hoạt động end-to-end.')

## 10. Visualize prediction trên mesh

Tô màu vertex theo class dự đoán — đây sẽ là cách bạn show kết quả trong báo cáo.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(
        x_in=feats,
        mass=mass_d, L=L_d,
        evals=evals_d, evecs=evecs_d,
        gradX=gradX_d, gradY=gradY_d,
        faces=faces_d,
    )
    pred = logits.argmax(dim=-1).cpu().numpy()

fig = go.Figure(data=[go.Mesh3d(
    x=verts_np[:, 0], y=verts_np[:, 1], z=verts_np[:, 2],
    i=faces_np[:, 0], j=faces_np[:, 1], k=faces_np[:, 2],
    intensity=pred, colorscale=[[0, 'lightblue'], [1, 'red']],
    showscale=True, opacity=1.0, flatshading=True,
)])
fig.update_layout(
    title='Prediction giả (xanh = original, đỏ = fracture)',
    scene=dict(aspectmode='data'),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

## Bước tiếp theo

Nếu notebook chạy hết không lỗi, bạn đã xong phần setup khó nhất. Tiếp theo:

1. **Tải Breaking Bad Dataset** (subset artifact) → upload thành Kaggle Dataset → attach vào notebook qua `/kaggle/input/...`.
2. **Viết Dataset class** đọc mesh + gán nhãn fracture/original (vertex nằm gần đường vỡ = 1).
3. **Vòng train thật** — DataLoader, train/val split, save checkpoint.
4. **Tính metrics**: Accuracy, Precision, Recall, F1, IoU per-vertex.
5. **Visualize kết quả** trên mesh test (giống cell 10).

Khi bạn xác nhận notebook này chạy xanh, báo lại để mình viết tiếp phần (1) và (2).